# **INFERENCE unsloth/gemma-3n-E2B-it**

configure colab environment for unsloth

In [ ]:
%%capture

import os
import re
import torch

v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)

xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, "0.0.34")

!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install torchcodec

torch._dynamo.config.recompile_limit = 64


In [ ]:
%%capture
!pip install --no-deps --upgrade timm

In [ ]:
from unsloth import FastModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import json
from google.colab import drive
drive.mount('/content/drive')

random.seed(42)
DRIVE_DIR = "/content/drive/MyDrive/TextMiningProject/"
DATASET = os.path.join(DRIVE_DIR, "data")
os.makedirs(DATASET, exist_ok=True)

Mounted at /content/drive


In [ ]:
ALL_DATA_JSON = os.path.join(DATASET, "all_data_processed.json")

with open(ALL_DATA_JSON, "r", encoding="utf-8") as f:
    all_data = json.load(f)
    samsum_data = [item for item in all_data if item["dataset"] == "samsum"]
    xsum_data   = [item for item in all_data if item["dataset"] == "xsum"]
    print(f"Loaded all_data from {ALL_DATA_JSON}")
    print(f"Total samples: {len(all_data)}")

Loaded all_data from /content/drive/MyDrive/TextMiningProject/data/all_data_processed.json
Total samples: 600


## Model: gemma-3n-E2B-it
https://ai.google.dev/gemma/docs/core/huggingface_inference?hl=it#vision

In [ ]:
!pip -q install modelscope

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 47.7 MB/s eta 0:00:00


In [ ]:
from unsloth.chat_templates import get_chat_template

In [ ]:
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/gemma-3n-E2B-it',
    dtype=None,
    load_in_4bit=True,
    token="token" # insert your token
)

==((====))==  Unsloth 2026.1.4: Fast Gemma3N patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3n won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/469M [00:00<?, ?B/s]

...

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

In [ ]:
SYS_PROMPT = "You are a helpful summarization assistant."
PROMPT_AUDIO = "Summarize this audio in one sentence."
PROMPT_TEXT = "Summarize this text in one sentence."

##### Decoding:
- The recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`.
Which is a "Sampling-based decoding" with nucleus sampling (top-p) combined with top-k sampling, which is a form of stochastic generation.
- Greedy decoding which choses everytime the token with the maximum probability
- Beam search decoding which is also the most computational expensive technique, which keeps tracks of the most promising k sequences computing the cumulative log-probabilities of the sequences.

In [ ]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.cache_size_limit = 512
import gc

def do_gemma_3n_batch_inference(model, tokenizer, batch_data, sys_prompt="", prompt="", max_new_tokens=64, decoding="temperature"):

    messages_batch = []
    if prompt == PROMPT_AUDIO:
        for item in batch_data:
            message = []
            message.append({
                "role": "system",
                "content": [{"type": "text", "text": sys_prompt}]
            })

            user_content = []
            user_content.append({"type": "audio", "audio": item['audio_path']})
            user_content.append({"type": "text", "text": prompt})
            message.append({"role": "user", "content": user_content})
            messages_batch.append(message)

        inputs = tokenizer.apply_chat_template(
            messages_batch,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to("cuda")

    elif prompt == PROMPT_TEXT:
        for item in batch_data:
            message = f"<system>{sys_prompt}</s> <user>{item['text']}</s> <user>{prompt}</s> <assistant>"
            messages_batch.append(message)

        inputs = tokenizer(
            text=messages_batch,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

    if decoding == "temperature":
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            do_sample=True
        )
    elif decoding == "greedy":
        output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    elif decoding == "beam":
        output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        early_stopping=True,
        do_sample=False
    )
    else:
        print("Decoding chosen is invalid")
        return


    results = []
    for i in range(len(messages_batch)):
        generated_text = tokenizer.decode(
            output_ids[i][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        results.append(generated_text)


    del inputs, output_ids
    torch.cuda.empty_cache()
    gc.collect()

    return results

#### gemma audio summarization

- temperature

In [ ]:
batch_size = 20

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Audio Summary (Gemma)"):
    batch = all_data[i:i+batch_size]

    results = do_gemma_3n_batch_inference(
        model=model,
        tokenizer=tokenizer,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_AUDIO,
        decoding="temperature"
    )

    for item, summary in zip(batch, results):
        item["summary_gemma_audio"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/gemma"
output_json = "/content/drive/MyDrive/TextMiningProject/gemma/audio_summaries_temperature.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"Saved results to {output_json}")

Generating Audio Summary (Gemma): 100%|██████████| 30/30 [15:50<00:00, 31.68s/it]

Saved results to /content/drive/MyDrive/TextMiningProject/gemma/audio_summaries_temperature.json


- greedy

In [ ]:
batch_size = 20

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Audio Summary (Gemma)"):
    batch = all_data[i:i+batch_size]

    results = do_gemma_3n_batch_inference(
        model=model,
        tokenizer=tokenizer,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_AUDIO,
        decoding="greedy"
    )

    for item, summary in zip(batch, results):
        item["summary_gemma_audio"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/gemma"
output_json = "/content/drive/MyDrive/TextMiningProject/gemma/audio_summaries_greedy.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Audio Summary (Gemma): 100%|██████████| 30/30 [09:47<00:00, 19.59s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/gemma/audio_summaries_greedy.json


- beam

In [ ]:
batch_size = 10

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Audio Summary (Gemma)"):
    batch = all_data[i:i+batch_size]

    results = do_gemma_3n_batch_inference(
        model=model,
        tokenizer=tokenizer,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_AUDIO,
        decoding="beam"
    )

    for item, summary in zip(batch, results):
        item["summary_gemma_audio"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/gemma"
output_json = "/content/drive/MyDrive/TextMiningProject/gemma/audio_summaries_beam.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Audio Summary (Gemma): 100%|██████████| 60/60 [21:10<00:00, 21.18s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/gemma/audio_summaries_beam.json


#### gemma text summarization

- temperature

In [ ]:
batch_size = 30

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Text Summary (Gemma)"):
    batch = all_data[i:i+batch_size]

    results = do_gemma_3n_batch_inference(
        model=model,
        tokenizer=tokenizer,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_TEXT,
        decoding="temperature"
    )

    for item, summary in zip(batch, results):
        item["summary_gemma_text"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/gemma"
output_json = "/content/drive/MyDrive/TextMiningProject/gemma/text_summaries_temperature.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Text Summary (Gemma): 100%|██████████| 20/20 [05:40<00:00, 17.01s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/gemma/text_summaries_temperature.json


- greedy

In [ ]:
batch_size = 30

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Text Summary (Gemma)"):
    batch = all_data[i:i+batch_size]

    results = do_gemma_3n_batch_inference(
        model=model,
        tokenizer=tokenizer,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_TEXT,
        decoding="greedy"
    )

    for item, summary in zip(batch, results):
        item["summary_gemma_text"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/gemma"
output_json = "/content/drive/MyDrive/TextMiningProject/gemma/text_summaries_greedy.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Text Summary (Gemma): 100%|██████████| 20/20 [05:23<00:00, 16.15s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/gemma/text_summaries_greedy.json


- beam

In [ ]:
batch_size = 20

for i in tqdm(range(0, len(all_data), batch_size), desc="Generating Text Summary (Gemma)"):
    batch = all_data[i:i+batch_size]

    results = do_gemma_3n_batch_inference(
        model=model,
        tokenizer=tokenizer,
        batch_data=batch,
        sys_prompt=SYS_PROMPT,
        prompt=PROMPT_TEXT,
        decoding="beam"
    )

    for item, summary in zip(batch, results):
        item["summary_gemma_text"] = summary

output_dir = "/content/drive/MyDrive/TextMiningProject/gemma"
output_json = "/content/drive/MyDrive/TextMiningProject/gemma/text_summaries_beam.json"

os.makedirs(output_dir, exist_ok=True)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to {output_json}")

Generating Text Summary (Gemma): 100%|██████████| 30/30 [10:03<00:00, 20.11s/it]


Saved results to /content/drive/MyDrive/TextMiningProject/gemma/text_summaries_beam.json
